In [1]:
from flodr import FloDR, viz
import numpy as np
import time
import scanpy as sc
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
from sklearn.neighbors import KNeighborsRegressor


In [2]:
# DP: dropped ds.get_paul15(), it fits HVG/scaling/PCA on all cells (leaks test into
# preprocessing). Your split below does it right.
difference = lambda x, y: np.sum(np.abs(x - y))


In [3]:
# DP: unchanged, well done!
sc.settings.verbosity = 0
adata = sc.datasets.paul15()
labels = adata.obs['paul15_clusters'].to_numpy()
sc.pp.filter_genes(adata, min_counts=1)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# split data into train/test and select highly variable genes
N, SEED = int(0.8 * len(adata.X)), 3
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(adata.X))
test_idx, train_idx = perm[N: ], perm[:N]

train_view = adata[train_idx, :]
sc.pp.highly_variable_genes(train_view, n_top_genes=1000)
hvg_mask = train_view.var.highly_variable

gt_log = adata[:,hvg_mask].copy()
train_log = adata[train_idx, hvg_mask].copy()
test_log = adata[test_idx, hvg_mask].copy()
genes = np.asarray(train_log.var_names)


In [4]:
from anndata import AnnData

def fit_preprocessor(train_adata:AnnData):
    """Fit scaling, PCA, and component standardization on training data."""
    # Scale (modifies train_adata in place)
    sc.pp.scale(train_adata, max_value=10)
    gene_mean = train_adata.var['mean'].values.astype(np.float64)
    gene_std  = train_adata.var['std'].values.astype(np.float64)

    train_x = np.asarray(train_adata.X, np.float64)   # scaled

    # PCA
    pca = PCA(n_components=50, svd_solver="randomized", random_state=0)
    pcs_train = pca.fit_transform(train_x)

    # Component-wise standardization
    pc_mean = pcs_train.mean(axis=0)
    pc_std  = pcs_train.std(axis=0) + 1e-9
    pcs_train_std = (pcs_train - pc_mean) / pc_std

    params = {
        'gene_mean': gene_mean,
        'gene_std': gene_std,
        'pca': pca,
        'pc_mean': pc_mean,
        'pc_std': pc_std
    }
    return pcs_train_std, params

def apply_forward(X_log:np.ndarray, params):
    """Apply scaling, PCA, and component standardization to log-normalised HVG data."""
    X_scaled = (X_log - params['gene_mean']) / params['gene_std']
    X_scaled = np.clip(X_scaled, None, 10)
    pcs = params['pca'].transform(X_scaled)
    return (pcs - params['pc_mean']) / params['pc_std']

def apply_inverse(Z_std:np.ndarray, params):
    """Standardized PCA-50 back to log1p gene expression.

    DP: It was correct, all I did was make the return space explicit. Stop at log1p, no
    expm1, even though the brief says four steps. See markdown below.
    """
    Z_raw = Z_std * params['pc_std'] + params['pc_mean']
    X_scaled = params['pca'].inverse_transform(Z_raw)
    return X_scaled * params['gene_std'] + params['gene_mean']   # log1p space

def clip_like_scale(X_log:np.ndarray, params):
    """Ground truth clipped the way sc.pp.scale clipped the training data.

    DP: the clip is in z-scored space and cannot be undone, so the brief scores against
    clipped data. You clipped the prediction at 10 in log1p units, which is the wrong space and the wrong
    side. z <= 10 is the same as X_log <= mean + 10*std per gene. This touches 0.08% of test
    values here, but it must be the ground truth that gets clipped.
    """
    temp = params['gene_mean'] + 10.0 * params['gene_std']
    return np.minimum(X_log, temp)


In [5]:
print(f"train {len(train_idx)} test {len(test_idx)}")

t0 = time.perf_counter()
model = FloDR(w=2.0, random_state=SEED, density=True, n_components=2)
train_std, params =  fit_preprocessor(train_log.copy())

# DP: fit() returns self, not latents (your train_z was the estimator object).
# n_components stays 2, the brief says released estimator only.
model.fit(train_std)
Y = model.embedding_

print(f"fit {time.perf_counter() - t0:.0f}s")
print(f"embedding {Y.shape}")
print(f"latent round trip, max abs error: {model.roundtrip_:.2e}")


train 2184 test 546
fit 131s
embedding (2184, 2)
latent round trip, max abs error: 9.54e-06


In [6]:
test_arr = np.asarray(test_log.X, np.float64)
test_std = apply_forward(test_arr.copy(), params)
test_emb = model.transform(test_std)

# zero-section: position -> profile, residual pinned to zero
tp_zero = apply_inverse(model.inverse_transform(test_emb), params)

# DP: the brief scores the conditional mean, not the zero-section. conditional_moments
# averages fiber samples and returns sigma too. to_input = _to_raw + your inverse chain,
# so the averaging happens in gene space.
to_genes = lambda z: apply_inverse(model._to_raw(z), params)
tp_cond, sigma_test = viz.conditional_moments(
    model.flow_, test_emb, to_input=to_genes, n_samples=64)

# ground truth, clipped like the training data
gt_test = clip_like_scale(test_arr, params)

print(test_emb.shape, tp_zero.shape, tp_cond.shape, gt_test.shape)
print(f"median sigma over held-out cells (log1p units, pooled over genes): {np.median(sigma_test):.3f}")


(546, 2) (546, 1000) (546, 1000) (546, 1000)
median sigma over held-out cells (log1p units, pooled over genes): 7.575


In [ ]:
def r2(gt, pred, method):
    # print(f"Method: {method}")
    marker_genes = ["Cd34", "Itga2b"]
    res={'method':method}
    for gene in marker_genes:
        idx = int(np.where(genes == gene)[0][0])
        r2 = r2_score(gt[:, idx], pred[:, idx])
        res[gene] = round(r2,3)
        # print(f"{gene}: R² = {r2:.3f}")
    return res


In [11]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import r2_score

def knn_baseline_from_ground_truth(ground_truth, train_idx, test_idx, params,
                                   n_components=50, n_neighbors=15):
    """
    ground_truth: (n_total, 1000) raw log1p HVG expression
    train_idx/test_idx: integer positions into ground_truth
    params: dict with gene_mean, gene_std, pca
    Returns preds_test (n_test, 1000) in clipped log1p space.
    """
    train_log = ground_truth[train_idx]   # raw log1p
    test_log  = ground_truth[test_idx]

    # 1. Scale using training params
    train_scaled = np.clip((train_log - params['gene_mean']) / params['gene_std'], -10, 10)
    test_scaled  = np.clip((test_log  - params['gene_mean']) / params['gene_std'], -10, 10)

    # 2. Raw PCA scores (unstandardized)
    train_pca = params['pca'].transform(train_scaled)[:, :n_components]
    test_pca  = params['pca'].transform(test_scaled)[:, :n_components]

    # 3. KNN in PCA space
    nn = NearestNeighbors(n_neighbors=n_neighbors)
    nn.fit(train_pca)
    _, idx = nn.kneighbors(test_pca)

    # 4. Target = clipped log1p of training cells (same as evaluation target)
    target_train = np.clip((train_log - params['gene_mean']) / params['gene_std'], -10, 10)
    target_train = target_train * params['gene_std'] + params['gene_mean']

    preds = target_train[idx].mean(axis=1)
    return preds

# Usage
ground_truth = gt_log.X.copy()
knn_preds = knn_baseline_from_ground_truth(ground_truth, train_idx, test_idx, params,
                                           n_components=50, n_neighbors=15)

# Clipped truth for test cells (already evaluated against your model)
true_clipped_knn = np.clip((ground_truth[test_idx] - params['gene_mean']) / params['gene_std'], -10, 10)
true_clipped_knn = true_clipped_knn * params['gene_std'] + params['gene_mean']

# R² per gene
r2_knn = np.mean([r2_score(true_clipped_knn[:, j], knn_preds[:, j]) for j in range(true_clipped_knn.shape[1])])
print("Mean R² (KNN unstandardized PCA):", r2_knn)


Mean R² (KNN unstandardized PCA): 0.08355815261519678


In [12]:
import numpy as np

def cluster_mean_baseline(ground_truth, train_idx, test_idx, labels, params):
    """
    ground_truth: (n_total, 1000) raw log1p HVG expression
    train_idx / test_idx: integer positions into ground_truth
    labels: (n_total,) cluster labels
    params: dict with gene_mean, gene_std
    Returns predictions for test cells in clipped log1p HVG space.
    """
    train_log = ground_truth[train_idx]   # raw log1p
    test_log  = ground_truth[test_idx]

    train_labels = labels[train_idx]
    test_labels  = labels[test_idx]

    # Clipped training targets (same as model evaluation target)
    train_target = np.clip((train_log - params['gene_mean']) / params['gene_std'], -10, 10)
    train_target = train_target * params['gene_std'] + params['gene_mean']

    # Mean per cluster from training cells
    cluster_means = {}
    for c in np.unique(train_labels):
        mask = train_labels == c
        cluster_means[c] = train_target[mask].mean(axis=0)

    # Fallback for unseen clusters
    global_mean = train_target.mean(axis=0)

    preds = np.zeros((len(test_labels), train_target.shape[1]), dtype=np.float64)
    for i, c in enumerate(test_labels):
        if c in cluster_means:
            preds[i] = cluster_means[c]
        else:
            preds[i] = global_mean

    return preds

# Clipped test truth
test_log = ground_truth[test_idx]
true_clipped_mean = np.clip((test_log - params['gene_mean']) / params['gene_std'], -10, 10)
true_clipped_mean = true_clipped_mean * params['gene_std'] + params['gene_mean']

cluster_preds = cluster_mean_baseline(ground_truth, train_idx, test_idx, labels, params)

from sklearn.metrics import r2_score
r2_cluster = np.mean([r2_score(true_clipped_mean[:, j], cluster_preds[:, j]) for j in range(true_clipped_mean.shape[1])])
print("Mean R² (cluster label mean):", r2_cluster)

Mean R² (cluster label mean): 0.07440280697126186


In [ ]:
mean = r2(true_clipped_mean, cluster_preds,"cluster mean")
knn = r2(true_clipped_knn, knn_preds, "knn")
flodr = r2(gt_test, tp_cond, "flodr")
import pandas as pd
pd.DataFrame([mean,knn,flodr])

Method: cluster mean
Method: knn
Method: flodr


TypeError: cannot concatenate object of type '<class 'dict'>'; only Series and DataFrame objs are valid